# Stage 1 — Build the device corpus

Parse the frozen 510(k) records into a clean device table and de-duplicate on K-number.
These are the **nodes** of the predicate graph.

Network: none — reads `snapshot/510k_raw.json.gz`.

In [ ]:
import gzip, json
import pandas as pd

with gzip.open("snapshot/510k_raw.json.gz", "rt") as f:
    raw = json.load(f)
manifest = json.load(open("snapshot/SNAPSHOT.json"))
print(f"loaded {len(raw)} raw records from snapshot dated {manifest['snapshot_date']}")

### Normalise fields and de-duplicate

A device listed under two product codes appears twice in the raw pull; we keep one row per
K-number and join its product codes with `|`.

In [ ]:
df = pd.DataFrame(raw)
df["k_number"] = df["k_number"].astype(str)
df["decision_date"] = pd.to_datetime(df["decision_date"], errors="coerce")
df["decision_year"] = df["decision_date"].dt.year

agg = (df.sort_values("decision_date")
         .groupby("k_number")
         .agg(device_name=("device_name", "first"),
              applicant=("applicant", "first"),
              product_codes=("product_code", lambda s: "|".join(sorted(set(s)))),
              decision_date=("decision_date", "first"),
              decision_year=("decision_year", "first"),
              statement_or_summary=("statement_or_summary", "first"))
         .reset_index())

import os
os.makedirs("data", exist_ok=True)
agg.to_csv("data/corpus.csv", index=False)

n_summary = (agg["statement_or_summary"] == "Summary").sum()
print(f"CHECKPOINT  unique devices {len(agg)} (exp {manifest['n_unique_devices']})")
print(f"CHECKPOINT  with Summary   {n_summary} (exp {manifest['n_with_summary']})")
print("NOTE: the 510(k) record has NO predicate field — edges are text-mined in Stage 3.")